# High-frequency detections: plots and agreement [^download]

[^download]: This notebook can be downloaded as **{nb-download}
repr(example_HF.ipynb) **.

Use this high-frequency annotation example to inspect a dataset, compare detection counts over time, summarize activity by hour of day, and assess annotator agreement and performance. The examples use the `DetectionData` plotting API; replace the sample file path and selections with your own APLOSE results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from pandas import Timedelta

from disclose.dataclass.detection_data import DetectionData

## Load data

Load the sample APLOSE results and reshape detections into one-minute bins. The `filename_format` describes the timestamps embedded in the audio filenames.

In [ ]:
config = {
    "detection_file": Path(r"_static/APOCADO_results.csv"),
    "timebin_new": Timedelta("60s"),
    "filename_format": "%Y-%m-%dT%H-%M-%S",
}

data = DetectionData.from_dict(config)
data.coordinates = (40, -2)

## Overview

The overview summarizes the labels and annotators present in the loaded data.

In [ ]:
data.overview(annotator=data.annotator)

## Detection counts over time

Plot one annotator's detections for several labels, grouped into one-hour bins. The axis helper formats the time ticks for readability.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

data.plot.histogram(
    ax=ax,
    annotator="annotator1",
    label=["Buzz", "Click", "Whistle"],
    bin_size=Timedelta("1h"),
    color=["red", "green", "blue"],
)
data.plot.set_ax(
    ax=ax,
    x_ticks_res=Timedelta("2h"),
    date_format="%H:%M",
)
plt.show()

## Compare labels

Plot each label on its own axis and set a shared y-axis range so the hourly counts can be compared directly.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 8))

lim = []
color = ["red", "green", "blue"]
for i, ax in enumerate(axs):
    data.plot.histogram(
        ax=ax,
        annotator="annotator1",
        label=data.label[i],
        bin_size=Timedelta("1h"),
        color=color[i],
    )
    data.plot.set_ax(
        ax=ax,
        x_ticks_res=Timedelta("2h"),
        date_format="%H:%M",
    )
    lim.append(axs[i].get_ylim()[-1])

for ax in axs:
    ax.set_ylim([0, int(max(lim))])
    ax.set_yticks(range(0, int(max(lim)), 4))

plt.show()

## Polar plot: detections by hour of day

The polar plot wraps the 24-hour day around a circle, with midnight at the top and time moving clockwise. Each bar shows the mean proportion of one-minute bins containing the selected label for that hour across the days in the dataset. This summary does not require recorder coordinates.

In [ ]:
polar_ax, hourly_proportion = data.plot.polar(
    annotator="annotator1",
    label="Whistle",
    color="tab:purple",
)
data.plot.feature(
    ax=polar_ax,
    rise_set=True,
)
plt.show()
hourly_proportion

## Detection performance

Compare two annotators' detections for the same label. The returned scores and confusion matrix quantify agreement with respect to the selected positive class.

In [ ]:
annotators = ["annotator1", "annotator2"]
labels = 2 * ["Whistle"]

score, confusion_matrix, classification = data.detection_perf(
    annotator=annotators,
    label=labels,
)

print(f"{'':#^60}")
print(f"{f'({annotators[0]}-{labels[0]}) | ({annotators[1]}-{labels[1]})':#^60}")
print(f"{'':#^60}\n")

print(f"{' Performance metrics ':#^60}")
print(
    f"{'Recall:':<30}{score.recall:>30.2f}\n"
    f"{'Precision:':<30}{score.precision:>30.2f}\n"
    f"{'F-Score:':<30}{score.f_score:>30.2f}"
)

print(f"{' Confusion matrix ':#^60}")
print(
    f"{'True positive:':<30}{confusion_matrix['true_positive']:>30}\n"
    f"{'False positive:':<30}{confusion_matrix['false_positive']:>30}\n"
    f"{'True negative:':<30}{confusion_matrix['true_negative']:>30}\n"
    f"{'False negative:':<30}{confusion_matrix['false_negative']:>30}\n"
)

## Inter-annotator agreement

Compare hourly detections from two annotators and view their agreement over the same time bins.

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2, gridspec_kw={"width_ratios": [3, 1]}, figsize=(10, 6)
)

annotators = ["annotator1", "annotator2"]
label = "Buzz"

data.plot.histogram(
    ax=ax1,
    annotator=annotators,
    label=label,
    bin_size=Timedelta("1h"),
)

data.plot.agreement(
    ax=ax2,
    annotator=annotators,
    label=label,
    bin_size=Timedelta("1h"),
)

data.plot.set_ax(
    ax=ax1,
    x_ticks_res=Timedelta("3h"),
    date_format="%H:%M",
)
data.plot.feature(
    ax=ax1,
    legend=False,
)

plt.show()